In [ ]:
# Import pandas, CSV file, and look at the raw data

import pandas as pd

EED_Raw = pd.read_csv(
    "data/acs_education_employment_disability_raw.csv",
    header=None
)

EED_Raw.head(40)

In [ ]:
# Create a separate copy for parsing the educational-attainment hierarchy

EED_ea = EED_Raw.copy()

EED_ea.columns = EED_ea.iloc[10]
EED_ea = EED_ea.iloc[11:].reset_index(drop=True)

In [ ]:
# Create a temporary copy of the entire dataframe
EED_Raw_ffill = EED_Raw.astype("object").copy()

# Forward-fill row 6 (Sex)
EED_Raw_ffill.iloc[6] = EED_Raw_ffill.iloc[6].ffill()
EED_Raw_ffill.iloc[6, 0] = "Sex"
EED_Raw_ffill.iloc[6, 1] = "Male"

# Forward-fill row 8 (Household language)
EED_Raw_ffill.iloc[8] = EED_Raw_ffill.iloc[8].ffill()
EED_Raw_ffill.iloc[8, 0] = "Household language"

# Display the updated temporary dataframe
EED_Raw_ffill.head(50)


# Remove ACS metadata rows that are not part of the analytical dataset
rows_to_drop = EED_Raw_ffill.iloc[[0, 1, 2, 3, 4, 5, 7, 9]].index
EED_Raw_ffill = EED_Raw_ffill.drop(rows_to_drop)

EED_Raw_ffill.head(50)

In [ ]:
# Transpose the table so demographic fields can be used as row-level identifiers
EED_Transposed = EED_Raw_ffill.T
EED_Transposed.head(75)

In [ ]:
# Split the nested ACS hierarchy into separate levels using "->" as the delimiter
EED_ea_split = EED_ea['Educational attainment (SCHL)'].astype(str).str.split('->', expand=True)

EED_ea_split = EED_ea_split.map(
    lambda x: x.strip() if isinstance(x, str) else x
)

EED_ea_split = EED_ea_split.replace('', pd.NA)

# Remove extra whitespace and convert blank hierarchy levels to missing values
EED_ea_split[[1, 2, 3, 4]] = EED_ea_split[[1, 2, 3, 4]].ffill()

# Carry parent hierarchy labels down to each educational-attainment category
EED_ea_split = EED_ea_split.dropna(subset=[0])

# Keep only rows that represent actual educational-attainment categories
EED_ea_split.columns = [
    'Educational attainment',
    'Level 1',
    'Level 2',
    'Level 3',
    'Level 4'
]

# Reconstruct the complete hierarchy path for validation

# Create a complete header path for each educational attainment row

EED_ea_split["header_path"] = (
    EED_ea_split["Level 1"] + " -> " +
    EED_ea_split["Level 2"] + " -> " +
    EED_ea_split["Level 3"] + " -> " +
    EED_ea_split["Level 4"]
)

# Reconstruct the complete hierarchy path for validation
EED_ea_split[["Educational attainment", "header_path"]].head(20)

In [ ]:
# Validate the expected hierarchy structure

print("Hierarchy rows:", EED_ea_split.shape[0])

print(
    "Unique hierarchy paths:",
    EED_ea_split["header_path"].nunique()
)

print(
    "Educational attainment categories:",
    EED_ea_split["Educational attainment"].nunique()
)

In [ ]:
# Build metadata for every column in EED_Transposed

column_metadata = pd.DataFrame({
    'column_position': EED_Transposed.columns,
    'header': EED_Transposed.iloc[0].values
})

# Identify and split columns containing hierarchical ACS labels

# Get the hierarchical/header columns
hierarchical_columns = column_metadata[
    column_metadata['header']
    .astype(str)
    .str.contains('->', regex=False)
].copy()

# Clean and split the paths
hierarchical_parts = (
    hierarchical_columns['header']
    .astype(str)
    .str.replace(r'^->\s*', '', regex=True)
    .str.split(' -> ')
)

hierarchy_metadata = pd.DataFrame(
    hierarchical_parts.tolist()
)

hierarchy_metadata.columns = [
    f'Level_{i+1}'
    for i in range(hierarchy_metadata.shape[1])
]

hierarchy_metadata.insert(
    0,
    'header_position',
    hierarchical_columns['column_position'].values
)

print(hierarchy_metadata.shape)
hierarchy_metadata.head()

In [ ]:
# Rename hierarchy levels using their analytical dimensions
hierarchy_metadata = hierarchy_metadata.rename(columns={
    'Level_2': 'Total',
    'Level_3': 'Employment Status',
    'Level_4': 'Disability Status',
    'Level_5': 'School Attendance'
})

# Remove the useless empty level
hierarchy_metadata = hierarchy_metadata.drop(columns='Level_1')

hierarchy_metadata.head()

# Extract the 25 educational-attainment categories
educational_attainment = EED_ea_split[
    'Educational attainment'
].unique().tolist()

print("Number of educational attainment categories:",
      len(educational_attainment))

print(educational_attainment)

# Keep only complete terminal hierarchy headers that define measurement blocks
terminal_hierarchy = hierarchy_metadata[
    hierarchy_metadata['School Attendance'].notna()
].copy()

print(terminal_hierarchy.shape)
terminal_hierarchy.head(10)

In [ ]:
# Each terminal hierarchy is followed by the same 25 educational-attainment categories
# Map those 25 columns to their corresponding hierarchy dimensions
measurement_metadata = []

for _, hierarchy in terminal_hierarchy.iterrows():

    header_position = hierarchy['header_position']

    for i, education in enumerate(educational_attainment, start=1):

        measurement_metadata.append({
            'column_position': header_position + i,
            'Educational Attainment': education,
            'Total': hierarchy['Total'],
            'Employment Status': hierarchy['Employment Status'],
            'Disability Status': hierarchy['Disability Status'],
            'School Attendance': hierarchy['School Attendance']
        })

measurement_metadata = pd.DataFrame(measurement_metadata)

print(measurement_metadata.shape)

In [ ]:
# Verify that every measurement column has exactly one metadata assignment
print(
    "Duplicate positions:",
    measurement_metadata['column_position'].duplicated().sum()
)

print(
    "Position range:",
    measurement_metadata['column_position'].min(),
    "to",
    measurement_metadata['column_position'].max()
)

measurement_metadata.head(30)

In [ ]:
# Select demographic identifiers and the mapped measurement columns

# Remove the header row so only actual data rows remain
data = EED_Transposed.iloc[1:].copy()

# Keep the demographic columns and the 1,400 measurement columns
measurement_positions = measurement_metadata['column_position'].tolist()

clean_values = data[
    [6, 8, 10] + measurement_positions
].copy()

# Give the demographic columns readable names
clean_values = clean_values.rename(columns={
    6: 'Sex',
    8: 'Household Language',
    10: 'Age / Total'
})

# Reshape the 1,400 measurement columns into a long-format value column
clean_values = clean_values.melt(
    id_vars=['Sex', 'Household Language', 'Age / Total'],
    var_name='column_position',
    value_name='Value'
)

print(clean_values.shape)
clean_values.head()

In [ ]:
# Attach the decoded hierarchy metadata to each population observation
EED_Clean = clean_values.merge(
    measurement_metadata,
    on='column_position',
    how='left'
)

print(EED_Clean.shape)
print(EED_Clean.columns.tolist())

EED_Clean.head()

In [ ]:
# Convert the age field to numeric and keep only ages 18–30
EED_Clean["Age"] = pd.to_numeric(
    EED_Clean["Age / Total"],
    errors="coerce"
)

EED_Clean = EED_Clean[
    EED_Clean["Age"].between(18, 30)
].copy()

EED_Clean["Age"] = EED_Clean["Age"].astype(int)

print(EED_Clean.shape)
print(EED_Clean["Age"].unique())

# Convert population estimates to numeric values
EED_Clean["Value"] = pd.to_numeric(
    EED_Clean["Value"],
    errors="coerce"
)

print("Value dtype:", EED_Clean["Value"].dtype)
print("Missing Values:", EED_Clean["Value"].isna().sum())

print("\nHousehold Language:")
print(
    EED_Clean["Household Language"]
    .value_counts(dropna=False)
)

In [ ]:
# Remove the "Total " prefix inherited from the ACS hierarchy
EED_Clean["Employment Status"] = (
    EED_Clean["Employment Status"]
    .str.removeprefix("Total ")
)

EED_Clean["Disability Status"] = (
    EED_Clean["Disability Status"]
    .str.removeprefix("Total ")
)

EED_Clean["School Attendance"] = (
    EED_Clean["School Attendance"]
    .str.removeprefix("Total ")
)

In [ ]:
# Confirm that age-inapplicable structural categories contain only zero population values

# Check the employment N/A category
employment_na = EED_Clean[
    EED_Clean["Employment Status"] == "N/A (less than 16 years old)"
]

print("Employment N/A rows:", len(employment_na))
print("Non-zero values:", (employment_na["Value"] != 0).sum())
print("Value sum:", employment_na["Value"].sum())


# Check the school-attendance N/A category
school_na = EED_Clean[
    EED_Clean["School Attendance"] == "N/A (less than 3 years old)"
]

print("\nSchool N/A rows:", len(school_na))
print("Non-zero values:", (school_na["Value"] != 0).sum())
print("Value sum:", school_na["Value"].sum())

In [ ]:
# Remove structural N/A categories that cannot apply to ages 18–30
EED_Clean = EED_Clean[
    (EED_Clean["Employment Status"] != "N/A (less than 16 years old)") &
    (EED_Clean["School Attendance"] != "N/A (less than 3 years old)")
].copy()

print(EED_Clean.shape)

# Confirm the Total hierarchy column is redundant
print(EED_Clean["Total"].unique())

# Remove temporary transformation fields that are no longer needed
EED_Clean = EED_Clean.drop(
    columns=["Age / Total", "column_position", "Total"]
)

# Arrange the final analytical schema
EED_Clean = EED_Clean[
    [
        "Sex",
        "Age",
        "Household Language",
        "Educational Attainment",
        "Employment Status",
        "Disability Status",
        "School Attendance",
        "Value"
    ]
]

In [ ]:
# Confirm population estimates are whole-number counts before converting to integers
print(
    "Fractional population values:",
    (EED_Clean["Value"] % 1 != 0).sum()
)

# Store population as integers and rename the value field to Population
EED_Clean["Value"] = EED_Clean["Value"].astype("int64")
EED_Clean = EED_Clean.rename(
    columns={"Value": "Population"}
)

In [ ]:
# Validate the final dataset structure, completeness, category counts, and population total

print("Shape:", EED_Clean.shape)

print(
    "Missing values:",
    EED_Clean.isna().sum().sum()
)

print(
    "Duplicate rows:",
    EED_Clean.duplicated().sum()
)

print(
    "Age range:",
    EED_Clean["Age"].min(),
    "-",
    EED_Clean["Age"].max()
)

print("\nCategory counts:")

print("Sex:", EED_Clean["Sex"].nunique())

print(
    "Household Language:",
    EED_Clean["Household Language"].nunique()
)

print(
    "Educational Attainment:",
    EED_Clean["Educational Attainment"].nunique()
)

print(
    "Employment Status:",
    EED_Clean["Employment Status"].nunique()
)

print(
    "Disability Status:",
    EED_Clean["Disability Status"].nunique()
)

print(
    "School Attendance:",
    EED_Clean["School Attendance"].nunique()
)

print(
    "\nTotal Population:",
    EED_Clean["Population"].sum()
)

In [ ]:
EED_Clean.to_csv(
    "acs_education_employment_disability_clean.csv",
    index=False
)